In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
    resume_data = pd.read_csv("/kaggle/input/resume-dataset/Resume/Resume.csv")
    resume_data

In [ ]:
pip install PyPDF2

In [ ]:
from pypdf import PdfReader
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = "".join(page.extract_text() for page in reader.pages)
    return text

In [ ]:
from nltk import pos_tag, sent_tokenize, word_tokenize
from nltk.corpus import stopwords
import string
import re
def preprocess_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    sentences = sent_tokenize(text)
    features = {'feature': ""}
    stop_words = set(stopwords.words("english"))
    for sent in sentences:
        if any(criteria in sent for criteria in ['skills', 'education']):
            words = word_tokenize(sent)
            words = [word for word in words if word not in stop_words]
            tagged_words = pos_tag(words)
            filtered_words = [word for word, tag in tagged_words if tag not in ['DT', 'IN', 'TO', 'PRP', 'WP']]
            features['feature'] += " ".join(filtered_words)
    return features

In [ ]:
def process_resume_data(df):
    id = df['ID']
    category = df['Category']
    text = extract_text_from_pdf(f"/kaggle/input/resume-dataset/data/data/{category}/{id}.pdf")
    features = preprocess_text(text)
    df['Feature'] = features['feature']
    return df

In [ ]:
    num_desc = 15
    resume_data = pd.read_csv("/kaggle/input/resume-dataset/Resume/Resume.csv")
    resume_data = resume_data.drop(["Resume_html"], axis=1)
    resume_data = resume_data.apply(process_resume_data, axis=1)
    resume_data = resume_data.drop(columns=['Resume_str'])
    resume_data.to_csv("/kaggle/working/resume_data.csv", index=False)

    job_description = pd.read_csv("/kaggle/input/resume-and-job-description/training_data.csv")
    job_description = job_description[["job_description", "position_title"]][:num_desc]
    job_description['Features'] = job_description['job_description'].apply(lambda x : preprocess_text(x)['feature'])

In [ ]:
resume_data


In [ ]:
categories = np.sort(resume_data['Category'].unique())
categories
# create new df for corpus and category
df_categories = [resume_data[resume_data['Category'] == category].loc[:, ['Feature', 'Category']] for category in categories]

In [ ]:
from wordcloud import WordCloud
def wordcloud(df):
    txt = ' '.join(txt for txt in resume_data['Feature'])
    wordcloud = WordCloud(
        height=2000,
        width=4000
    ).generate(txt)

    return wordcloud

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(32, 20))

for i, category in enumerate(categories):
    wc = wordcloud(df_categories[i])

    plt.subplot(5, 5, i + 1).set_title(category)
    plt.imshow(wc)
    plt.axis('off')
    plt.plot()

plt.show()
plt.close()

In [ ]:
def remove_extra_word(text):
    
    extra_word=['company', 'name', 'citi', 'state', 'work', 'manag', 'project'] # extra words
    words = text.split()  # Split the text into words
    
    # Filter out the extra words
    filter_word = [word for word in words if word not in extra_word]
    
    filter_text = ' '.join(filter_word)
    
    return filter_text


# apply resume_data['Cleaned_Resume']

resume_data['Feature']=resume_data['Feature'].apply(lambda x:remove_extra_word(x))

In [ ]:
plt.figure(figsize=(32, 20))

for i, category in enumerate(categories):
    wc = wordcloud(df_categories[i])

    plt.subplot(5, 5, i + 1).set_title(category)
    plt.imshow(wc)
    plt.axis('off')
    plt.plot()

plt.show()
plt.close()

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()

resume_data['Category']=encoder.fit_transform(resume_data['Category'])
resume_data.head()

In [ ]:
resume_data.Category.unique().shape

In [ ]:
# Split data into training and validation
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(resume_data['Feature'], resume_data['Category'], test_size=0.15, random_state=42, stratify=resume_data['Category'])


# Print the sizes of the split datasets
print("Train data size:", X_train.shape)
print("Validation data size:", X_test.shape)

In [ ]:
job_description

In [ ]:
job_desc = job_description.drop(['job_description', 'position_title'], axis=1)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(stop_words='english',max_features=800)
tfidf_train_vectors = tfidf.fit_transform(X_train)
tfidf_test_vectors =tfidf.transform(X_test)

tfidf_jobDesc_vectors = tfidf.fit_transform(job_desc['Features'])
tfidf_train_vectors

In [ ]:
tfidf_jobDesc_vectors_dense = tfidf_jobDesc_vectors.toarray()
tfidf_train_vectors_dense = tfidf_train_vectors.toarray()
tfidf_train_vectors_dense.shape, tfidf_jobDesc_vectors_dense.shape

In [ ]:
tfidf_test_vectors

In [ ]:
tfidf.get_feature_names_out()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
k = 5
result_df = pd.DataFrame(columns=['jobId', 'resumeId', 'similarity', 'domainResume', 'domainDesc'])
for i, job_desc_emb in enumerate(tfidf_jobDesc_vectors_dense):
    job_desc_id = i
    job_title = job_description['position_title'].iloc[i]

    # Compute cosine similarities between the current job description and all resumes
    similarities = cosine_similarity([job_desc_emb], tfidf_train_vectors )
    top_k_indices = np.argsort(similarities[0])[::-1][:k]
   
    # Extract the relevant information and add it to the result DataFrame
    for j in top_k_indices:
        resume_id = resume_data['ID'].iloc[j]
        work_domain = resume_data['Category'].iloc[j]
        similarity_score = similarities[0][j]
        
        result_df.loc[i+j] = [job_desc_id, resume_id, similarity_score, work_domain,job_title ]
        

# Sort the results by similarity score (descending)
result_df = result_df.sort_values(by='similarity', ascending=False)

In [ ]:
result_df.head()

In [ ]:
result_group=result_df.groupby("jobId")
result_group

In [ ]:
num_desc = 15
for i in range(num_desc):
    print()
    print("jobId---cosineSimilarity---domainResume---domainDesc")
    print(result_group.get_group(i).values[0])
    print()